# Multi-Layer Perceptron (MLP) by Hand with Python

In this lab, you will:

- Build a Multi-Layer Perceptron (MLP) from scratch using Python
- Understand forward propagation and backpropagation
- Implement the Adam optimizer manually
- Train a neural network for binary classification


A Multi-Layer Perceptron (MLP) is a type of neural network consisting of:

- **Input layer**: This layer receives the input features and passes them to the next layer.

- **Hidden layer(s)**: These layers perform computations on the input data. Each hidden layer applies a linear transformation followed by a non-linear activation function (e.g., ReLU, Sigmoid).

- **Output layer**: This layer produces the final output of the network, which can be a single value for binary classification or multiple values for multi-class classification.

Each layer performs:

$$Z=XW^T +b$$

Then applies the activation function:
$$A=σ(Z)$$

Where:

- $X$: Input features
- $W$: Weights
- $b$: Bias
- $σ$: Activation function (Sigmoid)

### Part 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd

### Part 2: Define Core Functions

**2.1. Sigmoid Activation Function**:

The sigmoid function converts values into range $(0, 1)$, which is useful for binary classification.

In [ ]:
# Activation
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

**2.2. Binary Cross Entropy Loss**:

To measure prediction error, we use binary cross-entropy loss, defined as:

$$L = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

Where:
- $N$: Number of samples
- $y_i$: True label (0 or 1)
- $\hat{y}_i$: Predicted probability

In [ ]:
# Loss
def binary_cross_entropy(y, y_hat):
    eps = 1e-9
    return -np.mean(y * np.log(y_hat + eps) +
     (1 - y) * np.log(1 - y_hat + eps))

**2.3. Adam Optimizer**:

The Adam optimizer performs adaptive gradient updates using first and second moment estimates of the gradients. The update rules are:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
# Adam Optimizer
def adam_update(W, b, dW, db, m_w, v_w, m_b, v_b,
                t, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):

    m_w = beta1 * m_w + (1 - beta1) * dW
    v_w = beta2 * v_w + (1 - beta2) * (dW ** 2)

    m_b = beta1 * m_b + (1 - beta1) * db
    v_b = beta2 * v_b + (1 - beta2) * (db ** 2)

    m_w_hat = m_w / (1 - beta1 ** t)
    v_w_hat = v_w / (1 - beta2 ** t)

    m_b_hat = m_b / (1 - beta1 ** t)
    v_b_hat = v_b / (1 - beta2 ** t)

    W -= lr * m_w_hat / (np.sqrt(v_w_hat) + eps)
    b -= lr * m_b_hat / (np.sqrt(v_b_hat) + eps)

    return W, b, m_w, v_w, m_b, v_b

### Part 3: Prepare Dataset

We use a sample dataset with only two labels (0 and 1) for binary classification.

In [ ]:
data = {
    "prio": [120,110,115,100,125,105,118,102,122,108],
    "static_prio": [120,110,115,100,120,105,118,102,120,108],
    "normal_prio": [120,110,115,100,125,105,118,102,122,108],
    "policy": [0,1,0,1,0,1,0,1,0,1],
    "vm_pgoff": [1024,4096,2048,8192,512,6000,1500,7000,900,5000],
    "class": [0,1,0,1,0,1,0,1,0,1]
}

df = pd.DataFrame(data)

X = df.drop(columns="class").values
y = df["class"].values

Normalize features to ensure stable training

In [ ]:
X = (X - X.mean(axis=0)) / X.std(axis=0)

### Part 4: Initialize the Model

We define the architecture of our MLP, including the number of layers and neurons. We also initialize weights and biases randomly.

In [ ]:
input_size = X.shape[1]
hidden_size = 4

np.random.seed(42)

W = np.random.randn(hidden_size, input_size) * 0.01
b = np.zeros(hidden_size)

W_out = np.random.randn(hidden_size) * 0.01
b_out = 0.0

### Part 5: Training Loop

Epoch is one complete pass through the training dataset. We perform forward propagation to compute predictions, calculate loss, and then use backpropagation to compute gradients and update weights using the Adam optimizer.

In [ ]:
epochs = 100
lr = 0.01
t = 1

In the training process, we will monitor the loss and accuracy to evaluate the model's performance. After training, we can test the model on a separate test set to assess its generalization ability.

In [ ]:
for epoch in range(epochs):

    # Forward Pass
    Z = np.dot(X, W.T) + b
    H = sigmoid(Z)

    z_out = np.dot(H, W_out) + b_out
    y_hat = sigmoid(z_out)

    # Loss
    loss = binary_cross_entropy(y, y_hat)

    # Backpropagation
    dz_out = y_hat - y
    dW_out = np.dot(H.T, dz_out) / len(y)
    db_out = np.mean(dz_out)

    dH = dz_out[:, None] * W_out
    dZ = dH * H * (1 - H)

    dW = np.dot(dZ.T, X) / len(X)
    db = np.mean(dZ, axis=0)

    # Initialize Adam optimizer variables on the first epoch
    if epoch == 0:
        m_w_out, v_w_out, m_b_out, v_b_out = 0, 0, 0, 0 # Scalars for W_out, b_out
        m_w, v_w = np.zeros_like(W), np.zeros_like(W)
        m_b, v_b = np.zeros_like(b), np.zeros_like(b)

    # Adam Updates
    W_out, b_out, m_w_out, v_w_out, m_b_out, v_b_out = adam_update(W_out, b_out, dW_out, db_out,
                                                                   m_w_out, v_w_out, m_b_out, v_b_out, t)

    W, b, m_w, v_w, m_b, v_b = adam_update(W, b, dW, db, m_w, v_w, m_b, v_b, t)

    t += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

### Part 6: Model Evaluation

After training, we make predictions on the entire dataset. We can evaluate the model's performance using the accuracy metric, which measures the proportion of correct predictions.

In [ ]:
Z = np.dot(X, W.T) + b
H = sigmoid(Z)

z_out = np.dot(H, W_out) + b_out
y_hat = sigmoid(z_out)

predictions = (y_hat >= 0.5).astype(int)

# Results
print("Probabilities:", y_hat)
print("Predictions:", predictions)
print("Actual:", y)

# Evaluation Metrics
accuracy = np.mean(predictions == y)
print(f"Accuracy: {accuracy:.4f}")

## Expected Results
- Loss decreases over epochs
- Predictions approach actual labels
- Model learns decision boundary

## Discussion Questions

1. Why do we use sigmoid instead of ReLU here?
2. What happens if we remove normalization?
3. Why is Adam better than standard gradient descent?
4. What does `dz_out = y_hat - y` represent?


## Extension Activities

Try improving the model:

1. Replace Sigmoid with ReLU
2. Add another hidden layer
3. Implement mini-batch gradient descent
4. Add dropout
5. Change the number of epochs
6. Compare with TensorFlow/Keras


## Key Takeaways
You implemented a full neural network from scratch and understand: Forward pass; Backpropagation; Gradient updates (Adam)

This is the foundation of deep learning frameworks